# Flusso di Lavoro Human-in-the-Loop con Microsoft Agent Framework

## 🎯 Obiettivi di Apprendimento

In questo notebook, imparerai come implementare flussi di lavoro **human-in-the-loop** utilizzando `RequestInfoExecutor` di Microsoft Agent Framework. Questo potente modello ti permette di mettere in pausa i flussi di lavoro AI per raccogliere input umano, rendendo i tuoi agenti interattivi e dando agli umani il controllo sulle decisioni critiche.

## 🔄 Cos'è Human-in-the-Loop?

**Human-in-the-loop (HITL)** è un modello di progettazione in cui gli agenti AI mettono in pausa l'esecuzione per richiedere input umano prima di continuare. Questo è essenziale per:

- ✅ **Decisioni critiche** - Ottenere l'approvazione umana prima di intraprendere azioni importanti
- ✅ **Situazioni ambigue** - Consentire agli umani di chiarire quando l'AI è incerta
- ✅ **Preferenze dell'utente** - Chiedere agli utenti di scegliere tra più opzioni
- ✅ **Conformità e sicurezza** - Garantire la supervisione umana per operazioni regolamentate
- ✅ **Esperienze interattive** - Costruire agenti conversazionali che rispondono all'input dell'utente

## 🏗️ Come Funziona in Microsoft Agent Framework

Il framework fornisce tre componenti chiave per HITL:

1. **`RequestInfoExecutor`** - Un esecutore speciale che mette in pausa il flusso di lavoro ed emette un `RequestInfoEvent`
2. **`RequestInfoMessage`** - Classe base per payload di richiesta tipizzati inviati agli umani
3. **`RequestResponse`** - Correlazione delle risposte umane con le richieste originali usando `request_id`

**Modello di Flusso di Lavoro:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Il Nostro Esempio: Prenotazione Hotel con Conferma Utente

Costruiremo sul flusso condizionale aggiungendo la conferma umana **prima** di suggerire destinazioni alternative:

1. L'utente richiede una destinazione (es. "Parigi")
2. `availability_agent` verifica se le camere sono disponibili
3. **Se nessuna camera** → `confirmation_agent` chiede "Vuoi vedere le alternative?"
4. Il flusso di lavoro **si mette in pausa** usando `RequestInfoExecutor`
5. **L'umano risponde** "sì" o "no" tramite input console
6. `decision_manager` instrada in base alla risposta:
   - **Sì** → Mostra destinazioni alternative
   - **No** → Annulla la richiesta di prenotazione
7. Mostra il risultato finale

Questo dimostra come dare agli utenti il controllo sulle proposte dell'agente!

---

Iniziamo! 🚀


## Passo 1: Importare le Librerie Necessarie

Importiamo i componenti standard del Framework Agent più le **classi specifiche per l'interazione umana**:
- `RequestInfoExecutor` - Esecutore che mette in pausa il flusso di lavoro per l'input umano
- `RequestInfoEvent` - Evento emesso quando viene richiesto un input umano
- `RequestInfoMessage` - Classe base per payload di richieste tipizzate
- `RequestResponse` - Correlazione delle risposte umane con le richieste
- `WorkflowOutputEvent` - Evento per rilevare le uscite del flusso di lavoro


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Passo 2: Definire i Modelli Pydantic per Output Strutturati

Questi modelli definiscono lo **schema** che gli agenti restituiranno. Conserviamo tutti i modelli del flusso di lavoro condizionale e aggiungiamo:

**Novità per Human-in-the-Loop:**
- `HumanFeedbackRequest` - Sottoclasse di `RequestInfoMessage` che definisce il payload della richiesta inviato agli umani
  - Contiene `prompt` (domanda da porre) e `destination` (contesto sulla città non disponibile)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Passo 3: Crea lo Strumento di Prenotazione Hotel

Stesso strumento del flusso di lavoro condizionale - verifica se ci sono camere disponibili nella destinazione.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Passo 4: Definire le Funzioni di Condizione per il Routing

Abbiamo bisogno di **quattro funzioni di condizione** per il nostro flusso di lavoro human-in-the-loop:

**Dal flusso di lavoro condizionale:**
1. `has_availability_condition` - Instrada quando gli hotel SONO disponibili
2. `no_availability_condition` - Instrada quando gli hotel NON sono disponibili

**Nuovo per human-in-the-loop:**
3. `user_wants_alternatives_condition` - Instrada quando l'utente dice "sì" alle alternative
4. `user_declines_alternatives_condition` - Instrada quando l'utente dice "no" alle alternative


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Passo 5: Crea il Decision Manager Executor

Questo è il **cuore del pattern human-in-the-loop**! Il `DecisionManager` è un `Executor` personalizzato che:

1. **Riceve feedback umano** tramite oggetti `RequestResponse`
2. **Elabora la decisione dell'utente** (sì/no)
3. **Instrada il flusso di lavoro** inviando messaggi agli agenti appropriati

Caratteristiche principali:
- Utilizza il decoratore `@handler` per esporre i metodi come passaggi del workflow
- Riceve `RequestResponse[HumanFeedbackRequest, str]` contenente sia la richiesta originale che la risposta dell'utente
- Produce semplici messaggi "sì" o "no" che attivano le nostre funzioni condizionali


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Passo 6: Crea un Executor di Visualizzazione Personalizzato

Lo stesso executor di visualizzazione del workflow condizionale - restituisce i risultati finali come output del workflow.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Passo 7: Carica le Variabili d'Ambiente

Configura il client LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Passo 8: Creare Agenti AI ed Esecutori

Creiamo **sei componenti del flusso di lavoro**:

**Agenti (inclusi in AgentExecutor):**
1. **availability_agent** - Controlla la disponibilità dell'hotel usando lo strumento
2. **confirmation_agent** - 🆕 Prepara la richiesta di conferma umana
3. **alternative_agent** - Suggerisce città alternative (quando l'utente dice sì)
4. **booking_agent** - Incoraggia la prenotazione (quando le stanze sono disponibili)
5. **cancellation_agent** - 🆕 Gestisce il messaggio di cancellazione (quando l'utente dice no)

**Esecutori Speciali:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` che mette in pausa il flusso di lavoro per l'input umano
7. **decision_manager** - 🆕 Esecutore personalizzato che instrada in base alla risposta umana (già definito sopra)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Passaggio 9: Costruire il Workflow con il Coinvolgimento Umano

Ora costruiamo il grafo del workflow con **indirizzamento condizionale** includendo il percorso con il coinvolgimento umano:

**Struttura del Workflow:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Connessioni Chiave:**
- `availability_agent → confirmation_agent` (quando non ci sono stanze)
- `confirmation_agent → prepare_human_request` (trasformare il tipo)
- `prepare_human_request → request_info_executor` (pausa per umano)
- `request_info_executor → decision_manager` (sempre - fornisce RequestResponse)
- `decision_manager → alternative_agent` (quando l'utente dice "sì")
- `decision_manager → cancellation_agent` (quando l'utente dice "no")
- `availability_agent → booking_agent` (quando sono disponibili stanze)
- Tutti i percorsi terminano a `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Passo 10: Esegui il Caso di Test 1 - Città SENZA Disponibilità (Parigi con Conferma Umana)

Questo test dimostra il **ciclo completo con intervento umano**:

1. Richiesta hotel a Parigi
2. verifica disponibilità agente → Nessuna camera
3. agente conferma crea domanda per l’umano
4. request_info_executor **sospende il flusso di lavoro** ed emette `RequestInfoEvent`
5. **L'applicazione rileva l’evento e chiede all’utente nella console**
6. L’utente digita "sì" o "no"
7. L’applicazione invia la risposta tramite `send_responses_streaming()`
8. decision_manager instrada in base alla risposta
9. Risultato finale mostrato

**Schema Chiave:**
- Usa `workflow.run_stream()` per la prima iterazione
- Usa `workflow.send_responses_streaming(pending_responses)` per le iterazioni successive
- Ascolta l’evento `RequestInfoEvent` per rilevare quando è necessario l’input umano
- Ascolta l’evento `WorkflowOutputEvent` per catturare i risultati finali


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Passo 11: Esegui il Caso di Test 2 - Città CON disponibilità (Stoccolma - Nessun input umano necessario)

Questo test dimostra il **percorso diretto** quando le camere sono disponibili:

1. Richiedi hotel a Stoccolma
2. availability_agent verifica → Camere disponibili ✅
3. booking_agent suggerisce la prenotazione
4. display_result mostra la conferma
5. **Nessun input umano richiesto!**

Il flusso di lavoro evita completamente il percorso con intervento umano quando le camere sono disponibili.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Punti Chiave e Best Practice per il Human-in-the-Loop

### ✅ Cosa Hai Imparato:

#### 1. **Pattern RequestInfoExecutor**
Il pattern human-in-the-loop nel Microsoft Agent Framework utilizza tre componenti chiave:
- `RequestInfoExecutor` - Mette in pausa il flusso di lavoro ed emette eventi
- `RequestInfoMessage` - Classe base per payload di richiesta tipizzati (eredita questa!)
- `RequestResponse` - Correlazione tra risposte umane e richieste originali

**Comprensione Critica:**
- `RequestInfoExecutor` NON raccoglie input direttamente - mette solo in pausa il flusso di lavoro
- Il codice della tua applicazione deve ascoltare `RequestInfoEvent` e raccogliere input
- Devi chiamare `send_responses_streaming()` con un dizionario che mappa `request_id` alla risposta dell'utente

#### 2. **Pattern di Esecuzione in Streaming**
```python
# Prima iterazione
stream = workflow.run_stream(initial_request)

# Iterazioni successive (dopo l'input umano)
stream = workflow.send_responses_streaming(pending_responses)

# Elabora sempre gli eventi
events = [event async for event in stream]
```

#### 3. **Architettura Event-Driven**
Ascolta eventi specifici per controllare il flusso di lavoro:
- `RequestInfoEvent` - È necessario input umano (flusso di lavoro in pausa)
- `WorkflowOutputEvent` - Risultato finale disponibile (flusso completato)
- `WorkflowStatusEvent` - Cambiamenti di stato (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, ecc.)

#### 4. **Esecutori Personalizzati con @handler**
Il `DecisionManager` dimostra come creare esecutori che:
- Usano il decoratore `@handler` per esporre metodi come passi del flusso di lavoro
- Ricevono messaggi tipizzati (es. `RequestResponse[HumanFeedbackRequest, str]`)
- Instradano il flusso di lavoro inviando messaggi ad altri esecutori
- Accedono al contesto tramite `WorkflowContext`

#### 5. **Instradamento Condizionale con Decisioni Umane**
Puoi creare funzioni condizionali che valutano le risposte umane:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Applicazioni nel Mondo Reale:

1. **Flussi di Approvazione**
   - Ottenere l'approvazione del manager prima di elaborare i report spese
   - Richiedere revisione umana prima di inviare email automatiche
   - Confermare transazioni di alto valore prima dell'esecuzione

2. **Moderazione dei Contenuti**
   - Segnalare contenuti discutibili per revisione umana
   - Chiedere ai moderatori di decidere sui casi limite
   - Escalare agli umani quando la fiducia dell'IA è bassa

3. **Servizio Clienti**
   - Lasciare che l'IA gestisca automaticamente domande di routine
   - Escalare problemi complessi ad agenti umani
   - Chiedere al cliente se vuole parlare con un umano

4. **Elaborazione Dati**
   - Chiedere agli umani di risolvere dati ambigui
   - Confermare interpretazioni IA di documenti poco chiari
   - Permettere agli utenti di scegliere tra più interpretazioni valide

5. **Sistemi Critici per la Sicurezza**
   - Richiedere conferma umana prima di azioni irreversibili
   - Ottenere approvazione prima di accedere a dati sensibili
   - Confermare decisioni in settori regolamentati (sanità, finanza)

6. **Agenti Interattivi**
   - Creare bot conversazionali che pongono domande di follow-up
   - Costruire wizard che guidano utenti in processi complessi
   - Progettare agenti che collaborano con gli umani passo dopo passo

### 🔄 Confronto: Con vs Senza Human-in-the-Loop

| Caratteristica | Flusso Condizionale | Flusso Human-in-the-Loop |
|---------|---------------------|---------------------------|
| **Esecuzione** | Singolo `workflow.run()` | Ciclo con `run_stream()` + `send_responses_streaming()` |
| **Input Utente** | Nessuno (completamente automatizzato) | Prompt interattivi via `input()` o UI |
| **Componenti** | Agent + Executor | + RequestInfoExecutor + DecisionManager |
| **Eventi** | Solo AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent, ecc. |
| **Pausa** | Nessuna pausa | Flusso in pausa su RequestInfoExecutor |
| **Controllo Umano** | Nessun controllo umano | Decisioni chiave prese dagli umani |
| **Caso d'Uso** | Automazione | Collaborazione e supervisione |

### 🚀 Pattern Avanzati:

#### Molteplici Punti di Decisione Umana
Puoi avere più nodi `RequestInfoExecutor` nello stesso flusso di lavoro:
```python
.add_edge(agent1, request_info_1)  # Prima decisione umana
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Seconda decisione umana
.add_edge(decision_manager_2, final_agent)
```

#### Gestione Timeout
Implementa timeout per risposte umane:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Predefinito all'opzione sicura
```

#### Integrazione UI Ricca
Invece di `input()`, integra con UI web, Slack, Teams, ecc.:
```python
if isinstance(event, RequestInfoEvent):
    # Invia notifica al canale preferito dall'utente
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Human-in-the-Loop Condizionale
Chiedi input umano solo in situazioni specifiche:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Invia al umano solo se la fiducia è bassa o il valore è alto
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Best Practice:

1. **Estendi Sempre RequestInfoMessage**
   - Fornisce sicurezza di tipo e validazione
   - Abilita contesto ricco per il rendering UI
   - Chiarisce l'intento di ogni tipo di richiesta

2. **Usa Prompt Descrittivi**
   - Includi contesto su cosa stai chiedendo
   - Spiega le conseguenze di ogni scelta
   - Mantieni le domande semplici e chiare

3. **Gestisci Input Inaspettato**
   - Valida le risposte degli utenti
   - Fornisci valori predefiniti per input non valido
   - Dai messaggi di errore chiari

4. **Traccia gli ID di Richiesta**
   - Usa la correlazione tra request_id e risposte
   - Non cercare di gestire manualmente lo stato

5. **Progetta per Non-Bloccante**
   - Non bloccare thread in attesa di input
   - Usa pattern async in tutto il sistema
   - Supporta istanze concorrenti di flusso di lavoro

### 📚 Concetti Correlati:

- **Middleware Agent** - Intercetta chiamate agent (notebook precedente)
- **Gestione Stato Workflow** - Persiste stato del flusso tra esecuzioni
- **Collaborazione Multi-Agent** - Combina human-in-the-loop con team di agenti
- **Architetture Event-Driven** - Costruisci sistemi reattivi con eventi

---

### 🎓 Congratulazioni!

Hai padroneggiato i flussi di lavoro human-in-the-loop con Microsoft Agent Framework! Ora sai come:
- ✅ Mettere in pausa i flussi di lavoro per raccogliere input umano
- ✅ Usare RequestInfoExecutor e RequestInfoMessage
- ✅ Gestire l’esecuzione in streaming con eventi
- ✅ Creare esecutori personalizzati con @handler
- ✅ Instradare flussi basati su decisioni umane
- ✅ Costruire agenti AI interattivi che collaborano con umani

**Questo è un pattern critico per costruire sistemi AI affidabili e controllabili!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
